# Tutorial: B2B Export Sales Workflow Demo

**Audience:** business, information-systems, and supply-chain students who want to understand the project's core commercial workflow.

**Prerequisites:** basic Python and familiarity with product cost, quotation, and follow-up concepts. No API key is required.

**Learning goals:**
- structure a fictional buyer inquiry without guessing missing facts;
- compare Gross Margin and Markup correctly;
- understand what the Streamlit application persists from inquiry to quotation and follow-up.

All companies, contacts, prices, and requirements in this notebook are fictional.

## Outline

1. Create a fictional inquiry.
2. Separate confirmed and missing quotation inputs.
3. Compare Gross Margin and Markup.
4. Review the persisted workflow relationships.
5. Complete a short exercise and check the answer.

In [ ]:
from __future__ import annotations

from decimal import Decimal, ROUND_HALF_UP

MONEY = Decimal("0.01")

def money(value: Decimal) -> Decimal:
    return value.quantize(MONEY, rounding=ROUND_HALF_UP)

## Step 1 - Structure a fictional buyer inquiry

The application keeps confirmed facts separate from missing information. This prevents a quotation from silently inventing a destination, specification, or payment term.

In [ ]:
inquiry = {
    "company": "Fictional Meridian Retail",
    "product": "A5 recycled-paper notebook",
    "specification": "160 pages, foil logo",
    "quantity": 2000,
    "destination": "Toronto, Canada",
    "required_delivery_time": None,
    "payment_requirement": None,
}

confirmed = {key: value for key, value in inquiry.items() if value not in (None, "")}
missing = [key for key, value in inquiry.items() if value in (None, "")]
{"confirmed": confirmed, "missing": missing}

## Step 2 - Compare Gross Margin and Markup

- **Gross Margin:** selling price = cost ÷ (1 − margin rate)
- **Markup:** selling price = cost × (1 + markup rate)

The same percentage produces different selling prices, so the selected method must be explicit.

In [ ]:
def selling_price(cost: Decimal, rate: Decimal, method: str) -> Decimal:
    if method == "gross_margin":
        if not Decimal("0") <= rate < Decimal("1"):
            raise ValueError("Gross Margin rate must be between 0 and 1.")
        return money(cost / (Decimal("1") - rate))
    if method == "markup":
        if rate < Decimal("0"):
            raise ValueError("Markup rate cannot be negative.")
        return money(cost * (Decimal("1") + rate))
    raise ValueError("Unknown pricing method.")

cost_cny = Decimal("30000")
rate = Decimal("0.25")
{
    "Gross Margin": selling_price(cost_cny, rate, "gross_margin"),
    "Markup": selling_price(cost_cny, rate, "markup"),
}

## Step 3 - Convert the selected quotation to USD

The project defines the exchange rate as **1 USD = X CNY**. Each quotation may override the workspace default.

In [ ]:
exchange_rate = Decimal("7.20")
gross_margin_total_cny = selling_price(cost_cny, rate, "gross_margin")
quote_total_usd = money(gross_margin_total_cny / exchange_rate)
unit_quote_usd = money(quote_total_usd / Decimal(inquiry["quantity"]))
{
    "total_quote_usd": quote_total_usd,
    "unit_quote_usd": unit_quote_usd,
    "exchange_rate_definition": "1 USD = 7.20 CNY",
}

## Step 4 - Understand the traceable workflow

The Streamlit application persists these relationships in SQLite:

`Customer → Inquiry → Quotation → Follow-up`

Session State carries temporary page context only. SQLite stores the long-term customer, inquiry, product, quotation, follow-up, and timeline IDs.

In [ ]:
workflow_relationships = {
    "inquiry": ["customer_id", "matched_product_id"],
    "quotation": ["customer_id", "inquiry_id", "product_id"],
    "follow_up": ["customer_id", "inquiry_id", "quotation_id"],
}
workflow_relationships

## Exercise

A fictional quotation has CNY 50,000 total cost and uses a 20% target rate. Predict the selling price under Gross Margin and Markup before running the answer cell.

**Common pitfall:** treating a 20% markup as a 20% Gross Margin. The formulas are not interchangeable.

**Optional extension:** after the repository is published, open this notebook in Colab, clone the repository, and replace the helper above with `services.quotation.calculate_quotation`.

In [ ]:
exercise_cost = Decimal("50000")
exercise_rate = Decimal("0.20")
{
    "Gross Margin answer": selling_price(exercise_cost, exercise_rate, "gross_margin"),
    "Markup answer": selling_price(exercise_cost, exercise_rate, "markup"),
}

## Run the full application after publishing

Once a GitHub remote exists, set `REPOSITORY_URL` below. Colab can then clone the project, install dependencies, run tests, and start Streamlit. The placeholder intentionally avoids inventing a repository URL.

In [ ]:
REPOSITORY_URL = ""  # Add the real GitHub URL after the repository is published.

if REPOSITORY_URL:
    print("Clone the repository, install requirements.txt, then run python -m pytest.")
else:
    print("Repository URL is intentionally unset until the project is published.")